# Proportional Allocation Model:
$X_{r_1 p_1 r_2 p_2}=\frac{X_{r_1 p_1 c_2}}{\sum_{r_1 \in c_1}X_{r_1 p_1 c_2}} \underbrace{L_{p_1 p_2}}_{\text{unknown}} \frac{X_{r_2 p_2}}{\sum_{p_2 } X_{r_2 p_2}L_{p_1 p_2}}  X_{c_1 p_1 r_2}$

---

In [ ]:
import pandas as pd
import numpy as np
import scipy as sp
import matplotlib.pyplot as plt
from pandas.api.types import CategoricalDtype
import math
import time
import random

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/CCL/Models/general_model


# Data manipulation
---

#### Merged data

In [ ]:
trade_data = pd.read_csv('data/trade_data.csv')
display(trade_data[['country_states','state_name']].drop_duplicates()['country_states'].value_counts())
print('Total number of states: ', len(trade_data['state_name'].unique()))
print('Total number of products: ', len(trade_data['product_name'].unique()))

Russia           85
United States    53
Spain            52
Japan            41
China            31
Brazil           27
Chile            16
Canada           13
Name: country_states, dtype: int64

Total number of states:  318
Total number of products:  5890


In [ ]:
hs4_hs6 = pd.read_csv('data/hs4_hs6_brazil.csv')
dict_hs4_hs6_names = pd.Series(hs4_hs6['HS4'].values, index=hs4_hs6['Product']).to_dict()
trade_data['product_name_hs4'] = trade_data["product_name"].map(dict_hs4_hs6_names)
trade_data['product_id_hs4'] = trade_data['product_id'].astype(str).str[:-2].astype(np.int64)
trade_data

,country_states,state_name,trade_flow_name,country_name,product_id,product_name,trade_value,product_name_hs4,product_id_hs4
0,United States,Alabama,Exports,Afghanistan,20714,Meat and edible offal: of fowls of the species...,7442320.00,Poultry Meat,207
1,United States,Alabama,Exports,Afghanistan,20727,"Meat and edible offal: of turkeys, cuts and of...",45354.00,Poultry Meat,207
2,United States,Alabama,Exports,Afghanistan,271019,Light petroleum distillates nes,24454.00,Refined Petroleum,2710
3,United States,Alabama,Exports,Afghanistan,320820,"Acrylic or vinyl polymer paint or varnish, non...",4557.00,Nonaqueous Paints,3208
4,United States,Alabama,Exports,Afghanistan,320890,"Polymer based paint, varnish in non-aqueous me...",15508.00,Nonaqueous Paints,3208
...,...,...,...,...,...,...,...,...,...
19113616,Chile,Valparaiso,Imports,Vietnam,621490,"Shawls, scarves, etc, of material nes, not knit",11622.20,Scarves,6214
19113617,Chile,Valparaiso,Imports,Vietnam,621710,Clothing accessories not knit,5588.30,Other Non-Knit Clothing Accessories,6217
19113618,Chile,Valparaiso,Imports,Vietnam,711719,Imitation jewellery nes of base metal includin...,3826.93,Imitation Jewellery,7117
19113619,Chile,Valparaiso,Imports,Vietnam,852990,"Parts for radio/tv transmit/receive equipment,...",2479.12,Broadcasting Accessories,8529


#### Filter Data

In [ ]:
# Remove regions under threshold
df_subnat_import = trade_data[trade_data['trade_flow_name']=='Imports'].loc[:,("state_name", "trade_value")].groupby(by='state_name').sum().reset_index().drop_duplicates()
df_subnat_export = trade_data[trade_data['trade_flow_name']=='Exports'].loc[:,("state_name", "trade_value")].groupby(by='state_name').sum().reset_index().drop_duplicates()

low_regions = np.unique(df_subnat_export[df_subnat_export['trade_value'] < 1e+09]['state_name'].values)

low_regions = np.unique(np.append(low_regions, np.unique(df_subnat_import[df_subnat_import['trade_value'] < 1e+09]['state_name'].values)))
trade_data = trade_data[~((trade_data['state_name'].isin(low_regions)))]

print("Number of Regions removed %d from %d.\n %s"%(len(low_regions),len(df_subnat_export['state_name'].unique()),low_regions))
print('Total number of states: ', len(trade_data['state_name'].unique()))

Number of Regions removed 49 from 318.
 ['ARKHANGELSK REGION' 'Acre' 'Arica Parinacota' 'Atacama'
 'CHECHEN REPUBLIC' 'CHUKOTKA AUTONOMOUS AREA' 'Ceuta' 'Coquimbo'
 'De Aysen' 'De Los Rios' 'De Magallanes Y Antartica Chilena' 'Del Maule'
 'Del Ñuble' 'IVANOVO REGION' 'JEWISH AUTONOMOUS REGION'
 'KABARDINO-BALKARIAN REPUBLIC' 'KAMCHATKA TERRITORY'
 'KARACHAYEVO-CIRCASSIAN REPUBLIC' 'KOMI REPUBLIC' 'KOSTROMA REGION'
 'KURGAN REGION' 'MAGADAN REGION' 'Melilla' 'NENETS AUTONOMOUS AREA'
 'Northwest Terr.' 'Nunavut' 'Paraíba' 'Prince Edward Is.'
 'REPUBLIC OF ADYGEYA' 'REPUBLIC OF ALTAI' 'REPUBLIC OF BURYATIA'
 'REPUBLIC OF CRIMEA' 'REPUBLIC OF DAGHESTAN' 'REPUBLIC OF INGUSHETIA'
 'REPUBLIC OF KALMYKIA' 'REPUBLIC OF MARI EL' 'REPUBLIC OF MORDOVIA'
 'REPUBLIC OF NORTH OSSETIA ? ALANIA' 'REPUBLIC OF SAKHA (YAKUTIA)'
 'REPUBLIC OF TUVA' 'Roraima' 'SEVASTOPOL - THE FEDERAL CITY' 'Sergipe'
 'Shiga' 'Shimane' 'Tarapaca' 'Yukon' 'Zamora' 'Ávila']
Total number of states:  269


# ${L_{p_1 p_2}}$ Prediction
---

#### Functions

In [ ]:
def get_candidate(product, is_imp_or_exp, df_imp_exp, param_countries_above_rca, param_sectors_above_rca):
#find imports for export product
  if(is_imp_or_exp == 'exp'):
    df_product = df_imp_exp[df_imp_exp['product'].astype(str) == product].sort_values(by='rca_exp', ascending=False)
    countries  = df_product[df_product['rca_exp'] > param_countries_above_rca ]['geography'].copy().values
    df_imp = df_imp_exp[df_imp_exp['geography'].isin(countries)][['product','geography','rca_imp']].copy()
    df_imp.rename(columns = {'rca_imp':'rca'}, inplace = True)
    return get_candidate_matrix(countries, df_imp, param_sectors_above_rca)
  #find exports for import product
  elif(is_imp_or_exp == 'imp'):
    df_product = df_imp_exp[df_imp_exp['product'].astype(str) == product].sort_values(by='rca_imp', ascending=False)
    countries  = df_product[df_product['rca_imp'] > param_countries_above_rca ]['geography'].copy().values
    df_exp = df_imp_exp[df_imp_exp['geography'].isin(countries)][['product','geography','rca_exp']].copy()
    df_exp.rename(columns = {'rca_exp':'rca'}, inplace = True)
    return get_candidate_matrix(countries, df_exp, param_sectors_above_rca)
  #error
  else:
    print("ERROR need to specify if it's an import or export product you want to analyze")

def get_candidate_matrix(locations, df_imp_exp, param_sectors_above_rca):
  matrix = pd.pivot_table(df_imp_exp, values='rca', index=['product'], columns='geography').fillna(0)
  matrix.columns = matrix.columns.astype(str)
  matrix['mean_rca']  = matrix.mean(axis=1)
  matrix['count > n'] = matrix[locations].gt(param_sectors_above_rca).sum(axis=1)
  matrix['variance']  = matrix[locations].var(axis=1)
  matrix['ecart_type'] = matrix.variance.map(np.sqrt)
  matrix['sum'] = matrix[locations].sum(axis=1)
  #matrix = matrix[matrix.variance < variance_threshold_sup]
  matrix = matrix.sort_values(by='count > n', ascending=False)

  return (matrix)

# Function that maps the ordering column_name of a df with numbers from 1-inf
def add_order(df_candidates, column_name='count > n'):
  results = []
  if(not df_candidates.empty):
    col = df_candidates[column_name]
    last_value = col[0]
    last_ord = 1
    for i in range(len(col)):
      if(col[i]!=last_value):
        last_value = col[i]
        last_ord = last_ord + 1
      results.append(last_ord)
  df_candidates['new_ranking'] = results
  return df_candidates

#### Models

In [ ]:
################################################################################################################################
###-----------------------------------------         Baseline Model            ----------------------------------------------###
### Description : uses Randomness                                                                                            ###
### Input:  Output product                                                                                                   ###
### Return: Returns 3 RANDOM inputs for the output product                                                                   ###
################################################################################################################################
def baseline_model(df_imp_exp, exported):
    possible_inputs = df_imp_exp['product'].unique()
    possible_inputs = np.delete(possible_inputs, np.where(possible_inputs==exported))
    cand1 = random.choice(possible_inputs)
    possible_inputs = np.delete(possible_inputs, np.where(possible_inputs==cand1))
    cand2 = random.choice(possible_inputs)
    possible_inputs = np.delete(possible_inputs, np.where(possible_inputs==cand2))
    cand3 = random.choice(possible_inputs)
    return (cand1,cand2,cand3)

In [ ]:
################################################################################################################################
###-------------------------------------------     Backward & Forward      --------------------------------------------------###
### Description : uses the SPECIALIZATION HYPOTHESIS                                                                         ###
### Input:  Output product                                                                                                   ###
### Return: Returns 3 inputs for the output product                                                                          ###
################################################################################################################################
def backward_and_forward(df_imp_exp, exported):
    # Parameters
    param_rca_location1 = 2
    param_rca_sectors1  = 1.5
    param_rca_location2 = 3.5
    param_rca_sectors2  = 2
    param_top_imports   = 3

    # Backward
    matrix_backward = get_candidate(str(exported), 'exp', df_imp_exp, param_rca_location1, param_rca_sectors1).head(240) # we only analyze the top 100/99 candidates
    matrix_backward = add_order(matrix_backward)
    if(exported in matrix_backward.index.values): # if same product candidate for imp and exp, delete
      matrix_backward = matrix_backward.drop(exported)
    for imported in matrix_backward.index.values:
      # Forward
      matrix_forward = get_candidate(str(imported), 'imp', df_imp_exp, param_rca_location2, param_rca_sectors2)
      matrix_forward = add_order(matrix_forward)
      if(exported in matrix_forward.index.values):
        forward_value = matrix_forward.loc[exported,'new_ranking']
      else:
        forward_value = matrix_forward['new_ranking'].max() + 1
      matrix_backward.loc[imported,'new_ranking'] = matrix_backward.loc[imported,'new_ranking'] + forward_value # update ranking
    matrix_backward = matrix_backward.sort_values(by='new_ranking', ascending=True)

    # Test Results
    if(len(matrix_backward) >= 3):
     cand1 = str(matrix_backward.index.values[0])
     cand2 = str(matrix_backward.index.values[1])
     cand3 = str(matrix_backward.index.values[2])
    elif(len(matrix_backward) == 2):
     cand1 = str(matrix_backward.index.values[0])
     cand2 = str(matrix_backward.index.values[1])
     cand3 = str('NaN')
    elif(len(matrix_backward) == 1):
     cand1 = str(matrix_backward.index.values[0])
     cand2 = str('NaN')
     cand3 = str('NaN')
    else :
     cand1 = str('NaN')
     cand2 = str('NaN')
     cand3 = str('NaN')

    return (cand1,cand2,cand3)

#### Main

In [ ]:
###--- Calculates RCAs import and export ---###

def data_manip(trade_data, product_name='product_name_hs4', trade_flow_name='trade_flow_name'):
  column_names = ['geography','product','value_imp','value_exp','geography_imp','geography_exp','product_imp','product_exp','rca_imp','rca_exp']
  df_trade = trade_data[['state_name', product_name, trade_flow_name,'trade_value']].groupby(by=['state_name', product_name, trade_flow_name]).sum().reset_index().copy()
  df_trade = df_trade.rename(columns={'state_name':'geography', product_name:'product'})

  df_sectors_imports = df_trade[df_trade[trade_flow_name]=='Imports'].drop(columns=[trade_flow_name])
  df_sectors_exports = df_trade[df_trade[trade_flow_name]=='Exports'].drop(columns=[trade_flow_name])
  df_sectors_imports = df_sectors_imports.rename(columns={'trade_value':'value_imp'})
  df_sectors_exports = df_sectors_exports.rename(columns={'trade_value':'value_exp'})

  df_imp_exp = pd.merge(df_sectors_imports, df_sectors_exports, on=['geography','product']).fillna(0)

  # RCA calculation
  df_imp_exp['geography_imp'] = df_imp_exp[['geography','value_imp']].groupby(by=['geography']).transform(np.sum)
  df_imp_exp['geography_exp'] = df_imp_exp[['geography','value_exp']].groupby(by=['geography']).transform(np.sum)

  df_imp_exp['product_imp'] = df_imp_exp[['product','value_imp']].groupby(by=['product']).transform(np.sum)
  df_imp_exp['product_exp'] = df_imp_exp[['product','value_exp']].groupby(by=['product']).transform(np.sum)

  df_imp_exp['rca_imp'] = (df_imp_exp['value_imp']/df_imp_exp['geography_imp'])/(df_imp_exp['product_imp']/df_imp_exp['value_imp'].sum())
  df_imp_exp['rca_exp'] = (df_imp_exp['value_exp']/df_imp_exp['geography_exp'])/(df_imp_exp['product_exp']/df_imp_exp['value_exp'].sum())

  return df_imp_exp

In [ ]:
###--------------------------------------------- MAIN ----------------------------------------------------###
# Result file
df_value_chains = pd.DataFrame(columns=['export', 'import 1', 'import 2', 'import 3'])
df_imp_exp = data_manip(trade_data, 'product_name_hs4', 'trade_flow_name')

wanted_hs4_inputs = ['Cars', 'Delivery Trucks','Electrical Ignitions','Combustion Engines', 'Rubber Thread', 'Fermented Milk Products', 'Leather of Other Animals', 'Synthetic Fabrics', 'Integrated Circuits',\
                     'Telephones', 'Computers', 'Refined Petroleum','Processed Tobacco','Corn','Jewellery','Petroleum Coke','Non-Knit Women\'s Shirts', 'Pig Iron', 'Electric Locomotives', 'Nuclear Reactors', 'Planes, Helicopters, and/or Spacecraft', 'Military Weapons']
wanted_hs6_inputs = ['Small Diesel Cars', 'Diesel Trucks 5-20 tonnes', 'Medium Sized Cars', 'Spark-ignition non chargeable Cars', \
                     'Electronic integrated circuits: processors and controllers, whether or not combined with memories, converters, logic circuits, amplifiers, clock and timing circuits, or other circuits',\
                     'Telephones for cellular networks or for other wireless networks', 'Automatic data processing machines: portable, weighing not more than 10kg, consisting of at least a central processing unit, a keyboard and a display',\
                     'Petroleum oils and products nes','Petroleum spirit for motor vehicles', 'Womens, girls blouses & shirts, of cotton, not knit',\
                     'Pig iron, non-alloy, <0.5% phosphorus','Rail locomotives powered by electric accumulators', 'Rail locomotives, externally electrically powered', 'Nuclear reactors', 'Helicopters of an unladen weight < 2,000 kg', 'Spacecraft: (including satellites) and suborbital and spacecraft launch vehicles', 'Fixed wing aircraft, unladen weight 2,000-15,000 kg',
                     'Cigarettes containing tobacco','Windows, French-windows, frames, of wood', 'Asphalt or similar material articles, in rolls', 'Cameras, single lens reflex, for roll film <= 35 mm', 'Microscopes, for photomicrography',\
                     'Medical apparatus using alpha, beta or gamma radiatio', 'Military weapons: artillery weapons (e.g. guns, howitzers, and mortars)']

for exported in (wanted_hs4_inputs):
    (cand1,cand2,cand3) = backward_and_forward(df_imp_exp, exported)
    df_value_chains = pd.concat([df_value_chains, pd.DataFrame([{'export':exported,
                                                                'import 1':cand1,
                                                                'import 2':cand2,
                                                                'import 3':cand3}])], axis=0, ignore_index=True)
df_value_chains

,export,import 1,import 2,import 3
0,Cars,Motor vehicles; parts and accessories (8701 to...,Electrical Lighting and Signalling Equipment,Seats
1,Delivery Trucks,Motor vehicles; parts and accessories (8701 to...,Electrical Lighting and Signalling Equipment,Padlocks
2,Electrical Ignitions,Electromagnets,Alkaline Metals,Non-Knit Men's Undergarments
3,Combustion Engines,Precious Metal Scraps,Non-Knit Active Wear,Engine Parts
4,Rubber Thread,Flax Fibers,Musical Instrument Parts,Glass Balls
5,Fermented Milk Products,Metal Stoppers,Aluminium Bars,Hot-Rolled Iron
6,Leather of Other Animals,Leather Machinery,Synthetic Tanning Extracts,Synthetic Filament Yarn Woven Fabric
7,Synthetic Fabrics,Looms,Jute Yarn,Processed Artificial Staple Fibers
8,Integrated Circuits,Disc Chemicals for Electronics,Machines and apparatus of a kind used solely o...,Oscilloscopes
9,Telephones,Integrated Circuits,LCDs,Printed Circuit Boards


# Proportional Allocation Model Examples

In [ ]:
# Setting up the data
trade_data = pd.read_csv('/data/trade_data.csv')

trade_export = trade_data[trade_data['trade_flow_name']=='Exports']
trade_import = trade_data[trade_data['trade_flow_name']=='Imports']

trade_export = trade_export.rename(columns = {'country_states':'export_country', 'state_name':'export_subnat', 'country_name':'import_country'}).drop('trade_flow_name', axis='columns')
trade_import = trade_import.rename(columns = {'country_states':'import_country', 'state_name':'import_subnat', 'country_name':'export_country'}).drop('trade_flow_name', axis='columns')

trade_data = pd.concat([trade_export, trade_import])

hs4_hs6 = pd.read_csv('/data/hs4_hs6_brazil.csv')
dict_hs4_hs6_names = pd.Series(hs4_hs6['HS4'].values, index=hs4_hs6['Product']).to_dict()
trade_data['product_name_hs4'] = trade_data["product_name"].map(dict_hs4_hs6_names)
trade_data['product_id_hs4'] = trade_data['product_id'].astype(str).str[:-2].astype(np.int64)

#### Examples

In [ ]:
r2 = "Barcelona"
c2 = 'Spain'
p2 = 'Cars'

p1 = "Engine Parts"
r1 = "Jiangsu Province"
c1 = 'China'

print(r1,' --> ',r2)
print(p1,' --> ',p2,' : ',(trade_data[(trade_data['export_subnat']==r1) & (trade_data['import_country']==c2) & (trade_data['product_name_hs4']==p1)]['trade_value'].sum() / trade_data[(trade_data['import_country']==c2) & (trade_data['export_country']==c1) & (trade_data['product_name_hs4']==p1)]['trade_value'].sum())\
* (trade_data[(trade_data['export_subnat']==r2) & (trade_data['product_name_hs4']==p2)]['trade_value'].sum()/trade_data[(trade_data['export_subnat']==r2)]['trade_value'].sum())\
* (trade_data[(trade_data['export_country']==c1)& (trade_data['import_subnat']==r2) & (trade_data['product_name_hs4']==p1)]['trade_value'].sum()),
  ' - ',
  (trade_data[(trade_data['export_subnat']==r1) & (trade_data['import_country']==c2) & (trade_data['product_name_hs4']==p1)]['trade_value'].sum() / trade_data[(trade_data['import_country']==c2) & (trade_data['export_country']==c1) & (trade_data['product_name_hs4']==p1)]['trade_value'].sum())\
* 100\
* (trade_data[(trade_data['export_country']==c1)& (trade_data['import_subnat']==r2) & (trade_data['product_name_hs4']==p1)]['trade_value'].sum()))

p1 = "LCDs"
r1 = "Osaka"
c1 = 'Japan'

p2 = 'Telephones'
r2 = "Guangdong Province"
c2 = 'China'

print(r1,' --> ',r2)
print(p1,' --> ',p2,' : ',(trade_data[(trade_data['export_subnat']==r1) & (trade_data['import_country']==c2) & (trade_data['product_name_hs4']==p1)]['trade_value'].sum() / trade_data[(trade_data['import_country']==c2) & (trade_data['export_country']==c1) & (trade_data['product_name_hs4']==p1)]['trade_value'].sum())\
* (trade_data[(trade_data['export_subnat']==r2) & (trade_data['product_name_hs4']==p2)]['trade_value'].sum()/trade_data[(trade_data['export_subnat']==r2)]['trade_value'].sum())\
* (trade_data[(trade_data['export_country']==c1)& (trade_data['import_subnat']==r2) & (trade_data['product_name_hs4']==p1)]['trade_value'].sum()),
  ' - ',
  (trade_data[(trade_data['export_subnat']==r1) & (trade_data['import_country']==c2) & (trade_data['product_name_hs4']==p1)]['trade_value'].sum() / trade_data[(trade_data['import_country']==c2) & (trade_data['export_country']==c1) & (trade_data['product_name_hs4']==p1)]['trade_value'].sum())\
* 100\
* (trade_data[(trade_data['export_country']==c1)& (trade_data['import_subnat']==r2) & (trade_data['product_name_hs4']==p1)]['trade_value'].sum()))


p1 = "Disc Chemicals for Electronics"
r1 = "Zhejiang Province"
c1 = 'China'

p2 = "Integrated Circuits"
r2 = "Texas"
c2 = 'United States'

print(r1,' --> ',r2)
print(p1,' --> ',p2,' : ',(trade_data[(trade_data['export_subnat']==r1) & (trade_data['import_country']==c2) & (trade_data['product_name_hs4']==p1)]['trade_value'].sum() / trade_data[(trade_data['import_country']==c2) & (trade_data['export_country']==c1) & (trade_data['product_name_hs4']==p1)]['trade_value'].sum())\
* (trade_data[(trade_data['export_subnat']==r2) & (trade_data['product_name_hs4']==p2)]['trade_value'].sum()/trade_data[(trade_data['export_subnat']==r2)]['trade_value'].sum())\
* (trade_data[(trade_data['export_country']==c1)& (trade_data['import_subnat']==r2) & (trade_data['product_name_hs4']==p1)]['trade_value'].sum()),
  ' - ',
  (trade_data[(trade_data['export_subnat']==r1) & (trade_data['import_country']==c2) & (trade_data['product_name_hs4']==p1)]['trade_value'].sum() / trade_data[(trade_data['import_country']==c2) & (trade_data['export_country']==c1) & (trade_data['product_name_hs4']==p1)]['trade_value'].sum())\
* 100\
* (trade_data[(trade_data['export_country']==c1)& (trade_data['import_subnat']==r2) & (trade_data['product_name_hs4']==p1)]['trade_value'].sum()))


p1 = "Motor vehicles; parts and accessories (8701 to 8705)"
r1 = "Ontario"
c1 = 'Canada'

p2 = "Cars"
r2 = "Hiroshima"
c2 = 'Japan'

print(r1,' --> ',r2)
print(p1,' --> ',p2,' : ',(trade_data[(trade_data['export_subnat']==r1) & (trade_data['import_country']==c2) & (trade_data['product_name_hs4']==p1)]['trade_value'].sum() / trade_data[(trade_data['import_country']==c2) & (trade_data['export_country']==c1) & (trade_data['product_name_hs4']==p1)]['trade_value'].sum())\
* (trade_data[(trade_data['export_subnat']==r2) & (trade_data['product_name_hs4']==p2)]['trade_value'].sum()/trade_data[(trade_data['export_subnat']==r2)]['trade_value'].sum())\
* (trade_data[(trade_data['export_country']==c1)& (trade_data['import_subnat']==r2) & (trade_data['product_name_hs4']==p1)]['trade_value'].sum()),
  ' - ',
  (trade_data[(trade_data['export_subnat']==r1) & (trade_data['import_country']==c2) & (trade_data['product_name_hs4']==p1)]['trade_value'].sum() / trade_data[(trade_data['import_country']==c2) & (trade_data['export_country']==c1) & (trade_data['product_name_hs4']==p1)]['trade_value'].sum())\
* 100\
* (trade_data[(trade_data['export_country']==c1)& (trade_data['import_subnat']==r2) & (trade_data['product_name_hs4']==p1)]['trade_value'].sum()))


p1 = "Machines and apparatus of a kind used solely or principally for the manufacture of semiconductor boules or wafers"
r1 = "Shandong Province"
c1 = 'China'

p2 = "Computers"
r2 = "Chongqing"
c2 = 'China'

Jiangsu Province  -->  Barcelona
Engine Parts  -->  Cars  :  1075826.2062951361  -  912627279.6626477
Osaka  -->  Guangdong Province
LCDs  -->  Telephones  :  144779772.2347312  -  120055918190.6913
Zhejiang Province  -->  Texas
Disc Chemicals for Electronics  -->  Integrated Circuits  :  215058.15330091663  -  556019436.568237
Ontario  -->  Hiroshima
Motor vehicles; parts and accessories (8701 to 8705)  -->  Cars  :  10686.699563839595  -  2594288.9428271423
